# 🚀 Artemis II Orion Telemetry — Enhanced Visualisation Draft

This notebook prototypes enhanced visualizations for the Artemis II Orion spacecraft trajectory (`target_id = -1024`).

### Key Enhancements Included:
1. **3D Earth Globe Mesh (Radius ~6371 km)**: Adds visual physical context to Earth center $(0,0,0)$.
2. **Moon Ephemeris Ingest & Trajectory**: Fetches Moon vectors from JPL Horizons (`301`) to show Earth-Moon system dynamics.
3. **Animated 3D Trajectory Scrubbing**: Enables interactive timeline playback.
4. **Velocity Vector Field Overlay**: Visualizes direction and magnitude of velocity along the orbit.
5. **Specific Orbital Energy & Keplerian Dynamics**: Computes orbital energy $\epsilon$ over time.

In [19]:
!pip install plotly astroquery

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from astroquery.jplhorizons import Horizons

print("Libraries successfully imported.")

Libraries successfully imported.


## 1. Fetch Moon Ephemeris (Target ID: 301)

We query JPL Horizons for the Moon (`target_id = 301`) over the same timeframe to overlay lunar coordinates in geocentric space.

In [20]:
# Fetch Moon ephemeris directly from JPL Horizons for comparison/overlay
start_str = "2026-04-02 02:00"
stop_str = "2026-04-10 00:00"
step_str = "1h"

print(f"Fetching Moon vectors (ID: 301) from JPL Horizons ({start_str} to {stop_str})...")
obj_moon = Horizons(id="301", location="500@399", epochs={"start": start_str, "stop": stop_str, "step": step_str})
vec_moon = obj_moon.vectors().to_pandas()

df_moon = pd.DataFrame({
    "epoch": pd.to_datetime(vec_moon["datetime_str"].str.replace("A.D.", "").str.strip()),
    "x_km": vec_moon["x"] * 149597870.7,
    "y_km": vec_moon["y"] * 149597870.7,
    "z_km": vec_moon["z"] * 149597870.7,
    "range_km": vec_moon["range"] * 149597870.7,
})

print(f"Fetched {len(df_moon)} Moon state vectors.")
df_moon.head()

Fetching Moon vectors (ID: 301) from JPL Horizons (2026-04-02 02:00 to 2026-04-10 00:00)...
Fetched 191 Moon state vectors.


,epoch,x_km,y_km,z_km,range_km
0,2026-04-02 02:00:00,-384665.922587,-80851.774667,-19700.133500,393564.450537
1,2026-04-02 03:00:00,-384033.031296,-84399.370422,-19977.942610,393705.144804
2,2026-04-02 04:00:00,-383367.626311,-87939.816640,-20254.029575,393845.621999
3,2026-04-02 05:00:00,-382669.800160,-91472.816637,-20528.372453,393985.870576
4,2026-04-02 06:00:00,-381939.647890,-94998.074987,-20800.949497,394125.878853


## 2. Fetch Orion Trajectory (-1024)

Query Orion vectors from JPL Horizons for prototype visualization.

In [21]:
print("Fetching Orion vectors (ID: -1024) from JPL Horizons...")
obj_orion = Horizons(id="-1024", location="500@399", epochs={"start": start_str, "stop": stop_str, "step": step_str})
vec_orion = obj_orion.vectors().to_pandas()

df_orion = pd.DataFrame({
    "epoch": pd.to_datetime(vec_orion["datetime_str"].str.replace("A.D.", "").str.strip()),
    "x_km": vec_orion["x"] * 149597870.7,
    "y_km": vec_orion["y"] * 149597870.7,
    "z_km": vec_orion["z"] * 149597870.7,
    "vx_kms": vec_orion["vx"] * 149597870.7 / 86400.0,
    "vy_kms": vec_orion["vy"] * 149597870.7 / 86400.0,
    "vz_kms": vec_orion["vz"] * 149597870.7 / 86400.0,
    "range_km": vec_orion["range"] * 149597870.7,
})
df_orion["speed_kms"] = np.sqrt(df_orion["vx_kms"]**2 + df_orion["vy_kms"]**2 + df_orion["vz_kms"]**2)

# Specific Orbital Energy: epsilon = v^2 / 2 - mu / r
MU_EARTH = 398600.4418  # km^3 / s^2
df_orion["specific_energy"] = (df_orion["speed_kms"]**2 / 2.0) - (MU_EARTH / df_orion["range_km"])

print(f"Fetched {len(df_orion)} Orion state vectors.")
df_orion.head()

Fetching Orion vectors (ID: -1024) from JPL Horizons...
Fetched 191 Orion state vectors.


,epoch,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,range_km,speed_kms,specific_energy
0,2026-04-02 02:00:00,-24602.597910,-14661.647173,-1313.590155,-1.807003,-3.851912,-0.334413,28670.145463,4.267824,-4.795820
1,2026-04-02 03:00:00,-29092.645925,-27111.217203,-2391.890992,-0.820545,-3.110829,-0.268743,39838.690838,3.228432,-4.793974
2,2026-04-02 04:00:00,-31024.033500,-37309.243469,-3274.577787,-0.300057,-2.578978,-0.222838,48633.251614,2.605920,-4.800637
3,2026-04-02 05:00:00,-31496.027069,-45813.874956,-4008.424334,0.017037,-2.160043,-0.186121,55740.275611,2.168114,-4.800673
4,2026-04-02 06:00:00,-31012.549198,-52942.404170,-4621.830702,0.239978,-1.809130,-0.155448,61530.786490,1.831585,-4.800712


## 3. Enhanced 3D Orbit Trajectory with 3D Earth Globe & Moon Overlay

Renders a scaled 3D Earth sphere ($R \approx 6371\text{ km}$), Moon trajectory, and Orion path.

In [22]:
# Build 3D Earth Surface Mesh
r_earth = 6371.0  # km
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, np.pi, 30)
x_earth = r_earth * np.outer(np.cos(u), np.sin(v))
y_earth = r_earth * np.outer(np.sin(u), np.sin(v))
z_earth = r_earth * np.outer(np.ones(np.size(u)), np.cos(v))

fig_3d = go.Figure()

# Add 3D Earth Globe Surface
fig_3d.add_trace(go.Surface(
    x=x_earth, y=y_earth, z=z_earth,
    colorscale="Blues",
    showscale=False,
    opacity=0.8,
    name="Earth Globe (R=6371km)"
))

# Add Orion Trajectory
fig_3d.add_trace(go.Scatter3d(
    x=df_orion["x_km"], y=df_orion["y_km"], z=df_orion["z_km"],
    mode="lines+markers",
    marker=dict(size=3, color=df_orion["speed_kms"], colorscale="Plasma", colorbar=dict(title="Speed (km/s)")),
    line=dict(color="#00D2FF", width=4),
    name="Orion Path (-1024)",
    hovertext=df_orion["epoch"].dt.strftime("%Y-%m-%d %H:%M UTC"),
    hovertemplate="<b>Orion</b><br>Epoch: %{hovertext}<br>X: %{x:,.0f} km<br>Y: %{y:,.0f} km<br>Z: %{z:,.0f} km<extra></extra>"
))

# Add Moon Trajectory
fig_3d.add_trace(go.Scatter3d(
    x=df_moon["x_km"], y=df_moon["y_km"], z=df_moon["z_km"],
    mode="lines",
    line=dict(color="gray", width=3, dash="dash"),
    name="Moon Orbit (301)",
    hovertext=df_moon["epoch"].dt.strftime("%Y-%m-%d %H:%M UTC"),
    hovertemplate="<b>Moon</b><br>Epoch: %{hovertext}<br>Range: %{x:,.0f} km<extra></extra>"
))

# Current Moon Position
fig_3d.add_trace(go.Scatter3d(
    x=[df_moon["x_km"].iloc[-1]], y=[df_moon["y_km"].iloc[-1]], z=[df_moon["z_km"].iloc[-1]],
    mode="markers+text",
    marker=dict(size=8, color="lightgray"),
    name="Moon Position",
    text=["Moon"], textposition="top center"
))

fig_3d.update_layout(
    title="Enhanced Geocentric Trajectory: Orion & Moon with 3D Earth Globe",
    scene=dict(
        xaxis_title="X (km)",
        yaxis_title="Y (km)",
        zaxis_title="Z (km)",
        aspectmode="data"
    ),
    height=750,
    margin=dict(l=0, r=0, t=40, b=0)
)
fig_3d.show()

## 4. Specific Orbital Energy Profile ($\epsilon$)

Plots specific orbital energy $\epsilon = \frac{v^2}{2} - \frac{\mu}{r}$ over time.

In [23]:
fig_energy = px.line(
    df_orion,
    x="epoch",
    y="specific_energy",
    labels={"epoch": "Time (UTC)", "specific_energy": "Specific Energy (km²/s²)"},
    title="Orion Specific Orbital Energy (ε) over Time"
)
fig_energy.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Parabolic Escape Threshold (ε = 0)")
fig_energy.update_traces(line_color="#00CC96", line_width=2.5)
fig_energy.update_layout(height=450)
fig_energy.show()